# 04 — Google Play Store EDA and Feature Engineering Notes

### Clean a messy real-world app dataset and turn it into useful insights

**Level:** beginner → advanced  
**Based on:** the supplied Google Play Store lecture notebook and transcript.

> **Simple picture:** app-store data is like a shelf full of boxes with different labels. Some labels are numbers wearing costumes such as `1,000+` or `$4.99`. We remove the costumes before we count and compare.

## Learning goals

You will learn to clean mixed-format columns, handle duplicates, explore categories and installs, and create model-ready features without losing the meaning of the original data.


## 1. Why this dataset needs careful cleaning

The Google Play Store dataset mixes numbers and text:

- `Reviews` may contain a bad text value.
- `Size` can use `M`, `k`, or `Varies with device`.
- `Installs` can look like `1,000+`.
- `Price` can look like `$4.99`.
- `Last Updated` is a date written as text.

These are not mistakes by pandas. They are signals that we need to explain the data to the computer before doing maths with it.


In [ ]:
# Beginner-friendly guide:
# These libraries help us load the app table, clean string columns, and draw simple EDA charts.
# Keep a local CSV beside this notebook so the lesson runs without depending on an internet connection.
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

plt.style.use("seaborn-v0_8-whitegrid")


In [ ]:
# Beginner-friendly guide:
# Change this filename if your Google Play Store CSV has a different name or folder.
# We make a copy immediately so the raw download can stay untouched as a source file.
data_path = Path("googleplaystore.csv")
if not data_path.exists():
    raise FileNotFoundError("Add googleplaystore.csv beside this notebook, then run this cell again.")

apps = pd.read_csv(data_path).copy()
print("Shape:", apps.shape)
display(apps.head())
display(apps.isna().sum().rename("missing_count"))


## 2. Clean number-looking text

The goal is to keep the original meaning while changing the storage type.

| Raw text | Clean numeric idea |
| --- | --- |
| `"1,000+"` installs | 1000 installs or a lower-bound proxy. |
| `"$4.99"` price | 4.99 as a number. |
| `"19M"` size | 19 megabytes. |
| `"14k"` size | 14/1024 megabytes. |
| `"Varies with device"` | Missing numeric size. |

### Gentle warning

Turning `1,000+` into 1000 is useful for exploration, but it is a lower bound, not the exact install count.


In [ ]:
# Beginner-friendly guide:
# We safely convert reviews, installs, and price into numbers by removing symbols first.
# errors="coerce" turns unexpected text into missing values instead of crashing the notebook.
apps["Reviews"] = pd.to_numeric(apps["Reviews"], errors="coerce")
apps["Installs"] = pd.to_numeric(
    apps["Installs"].astype("string").str.replace("+", "", regex=False).str.replace(",", "", regex=False),
    errors="coerce",
)
apps["Price"] = pd.to_numeric(
    apps["Price"].astype("string").str.replace("$", "", regex=False),
    errors="coerce",
)

display(apps[["Reviews", "Installs", "Price"]].head())


In [ ]:
# Beginner-friendly guide:
# This helper converts app size into megabytes. A size that varies by device becomes missing because one fixed number would be dishonest.
# The small function keeps the cleaning rule in one clear place instead of repeating it many times.
def size_to_mb(value):
    text = str(value).strip().lower()
    if text in {"nan", "varies with device"}:
        return np.nan
    if text.endswith("m"):
        return pd.to_numeric(text[:-1], errors="coerce")
    if text.endswith("k"):
        return pd.to_numeric(text[:-1], errors="coerce") / 1024
    return pd.to_numeric(text, errors="coerce")

apps["size_mb"] = apps["Size"].apply(size_to_mb)
apps["Last Updated"] = pd.to_datetime(apps["Last Updated"], errors="coerce")
apps["last_updated_year"] = apps["Last Updated"].dt.year

display(apps[["Size", "size_mb", "Last Updated", "last_updated_year"]].head())


## 3. Duplicates and meaningful rows

An app can appear more than once because it was updated, listed in more than one category, or copied by mistake. Before removing duplicates, decide what one row should represent.

For a simple app-level study, keeping one record per `App` can be reasonable. If dates matter, you may prefer to keep the latest record instead of just the first one.

### Good question

“How many rows are duplicated?” is useful. “Which app records are duplicated, and why?” is better.


In [ ]:
# Beginner-friendly guide:
# We count repeated app names, then keep one row per app for a simple app-level analysis.
# Keep a separate copy so you can return to the uncollapsed table if your question needs every row.
print("Rows with a repeated app name:", apps.duplicated(subset="App").sum())

apps_one_row = apps.drop_duplicates(subset="App", keep="first").copy()
print("Rows after keeping one record per app:", len(apps_one_row))


## 4. Ask useful EDA questions

Good EDA questions for this dataset include:

- Which categories contain the most apps?
- Which categories have the largest total installs?
- Are higher-rated apps also more reviewed?
- How do free and paid apps differ in price, installs, or ratings?
- Which apps or categories create the long tail of installs?

Use both counts and proportions. A large category may have many apps but not necessarily the most installs.


In [ ]:
# Beginner-friendly guide:
# The first chart counts how many apps sit in each category.
# The second chart adds installs by category, which answers a different question: where is the biggest audience?
top_categories = apps_one_row["Category"].value_counts().head(10).sort_values()
category_installs = apps_one_row.groupby("Category")["Installs"].sum().sort_values().tail(10)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
top_categories.plot.barh(ax=axes[0], title="Top categories by app count")
category_installs.plot.barh(ax=axes[1], title="Top categories by total installs")

axes[0].set_xlabel("Number of apps")
axes[1].set_xlabel("Total installs (lower-bound proxy)")
plt.tight_layout()


In [ ]:
# Beginner-friendly guide:
# We compare rating with review count on a log scale because review counts can be extremely spread out.
# The hue colour lets us see whether free and paid apps form different clouds of points.
plot_data = apps_one_row.dropna(subset=["Rating", "Reviews", "Type"]).copy()
plot_data = plot_data[plot_data["Reviews"] > 0]

sns.scatterplot(data=plot_data, x="Reviews", y="Rating", hue="Type", alpha=0.55)
plt.xscale("log")
plt.title("Rating versus review count")
plt.show()


## 5. From exploration to model features

Possible model-ready features include:

- numeric: rating, reviews, installs, price, size in MB, update year;
- category: app type, content rating, genres, main category;
- date-derived: update year or app age;
- log-transformed: installs or reviews when a few huge values dominate.

### Advanced safety note

If the task is to predict a future rating or future installs, do not use information collected after the prediction date. That would be data leakage.

## End-of-topic recap

Clean the text costumes first, keep the meaning of each column clear, remove duplicates only with a reason, and use charts to answer questions rather than just decorate the notebook.
